# Learning Algorithms

> Inspired by [Sam Lau](https://www.samlau.me/), who co-authored the [Learning Data Science](https://learningds.org/ch/12/pa_intro.html) book.

In this lesson, we'll introduce machine learning from the ground up. By the end of this lesson, students will be able to:

- Describe the difference between traditional algorithms and machine learning algorithms.
- Identify the components of a machine learning model and dataset features and labels.
- Apply `sklearn` to train a linear regression model.

A while back, we discussed data visualization using the [Puget Sound Clean Air Agency's](https://pscleanair.gov/) EPA-grade air quality sensors (**AQS**). However, these sensors are typically expensive, costing anywhere between \\$15,000 and \\$40,000 each, making it hard to deploy a large number of these sensors. Furthermore, EPA-grade AQS measurements also undergo calibration and accuracy checks that lead to delays of one or two hours, leading to data that is very accurate but not necessarily timely.

In contrast, "[PurpleAir](https://www2.purpleair.com/) makes sensors that empower Community Scientists who collect hyper-local air quality data and share it with the public." In this lesson, we'll learn how we can use more accurate but less timely AQS measurements to calibrate the less accurate but more timely PurpleAir sensor (**PAS**) measurements so that we can provide the best information to the general public. The concepts in this lesson are actually used in the real world when you visit the EPA [AirNow Fire and Smoke Map](https://fire.airnow.gov/): the PAS data in this map are calibrated using the approach we will learn today.

In [ ]:
import pandas as pd
import seaborn as sns

sns.set_theme()

Our dataset includes over 12,000 matched observations where we've paired each AQS measurement with a nearby PAS measurement, along with 3 other variables that experts have identified as potentially impacting PAS measurement quality. The dataset includes 5 columns:

- The very accurate EPA-grade air quality sensor (AQS) measurement of the PM2.5 value.
- The temperature in degrees celsius.
- The relative humidity as a percentage between 0% and 100%.
- The dew point, where a higher dew point means more moisture in the air.
- The less accurate but more timely PurpleAir sensor (PAS) measurement of the PM2.5 value.

**How can we use the PAS measurement together with the temperature, humidity, and dew point to predict the AQS measurement?**

In [ ]:
sensor_data = pd.read_csv("sensor_data.csv")
sensor_data

Below we use data visualization to understand this dataset by scatter-plotting PAS to AQS. 

Why do we place PAS on the x-axis and AQS on the y-axis?

*TODO*: Enter your response here

In [ ]:
# Predict AQS from PAS (use cheaper PAS sensor to estimate more expensive AQS sensor)
feature = "PAS"
target  = "AQS"

sns.relplot(sensor_data, x=feature, y=target)

## Guessing game

Let's start with a simpler task: **How can we use the PAS measurement alone to predict the AQS measurement?** To do this, let's choose a line that best describes the trend, or the regression line that is produced by calling `lmplot`.

| R² Value | Interpretation  |
| -------- | --------------- |
| 0.0–0.3  | Weak fit        |
| 0.3–0.6  | Moderate fit    |
| 0.6–0.8  | Strong fit      |
| 0.8–1.0  | Very strong fit |


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import linregress

def plot_line(slope, intercept=0):
    grid = sns.relplot(sensor_data, x=feature, y=target)
    grid.facet_axis(0, 0).plot([0, 120], [intercept, slope * 120 + intercept], c="orange")
    grid.set(title=f"Slope = {slope:.2f}, intercept = {intercept:.2f}")
    return grid

plot_line(0.7, 5)  # adjust numbers manually to try to fit here...

What differentiates *machine learning* from just human guessing is the use of an algorithm to find the best line, which requires a **metric** for the quality of a trend.

We can visualize all of our guesses so far by plotting them against their mean squared errors on what's called a **loss surface**.

In [ ]:
from sklearn.metrics import mean_squared_error

def plot_loss(slopes, intercept=0):
    losses = [mean_squared_error(sensor_data[feature] * s, sensor_data[target]) for s in slopes]
    grid = sns.relplot(x=slopes, y=losses)
    grid.set(title="Loss surface", xlabel="Slope", ylabel="MSE", xlim=[0, 1], ylim=[0, None])
    return grid

# plot_loss([...]) # TODO: fill in your slope guesses to try

## Gradient descent

So how do we write a **machine learning algorithm** that can optimize this metric and find the minimum mean squared error in the loss surface so that it selects the best possible line? Machine learning scientists can apply concepts from linear algebra to solve this system by selecting a random initial theta (slope) value and then rolling down the hill toward the minimum value at the bottom of the bowl. 

$$
\nabla_{\!\theta}\; \text{MSE}(\boldsymbol{\theta}, \mathbf{X}, \mathbf{y}) = -\frac{2}{n}(\mathbf{X}^\top \mathbf{y} - \mathbf{X}^\top \mathbf{X} \boldsymbol{\theta})
$$

### 3Blue1Brown Gradient Descent

3Blue1Brown has a great video review of this process.

> 🎬 **Watch:** [3Blue1Brown — Gradient descent, how neural networks learn](https://www.youtube.com/watch?v=IHZwWFHWa-w) *(Focus on the loss surface and rolling downhill concepts)*

Answer the following questions in 1–2 sentences each:

**Loss Surface Intuition:** What does the height (y-axis) of a loss surface represent when evaluating our slope choice ($\theta$)?
   * *Your Answer:* 

**Learning Rate ($\alpha$):** Why do we multiply the gradient by a small learning rate (like $\alpha = 0.002$) instead of just subtracting the raw gradient itself (a "full step," $\alpha = 1$)?
   * *Your Answer:* 

**Update Direction:** In the update formula $\theta_{t+1} = \theta_t - \alpha \cdot \nabla \text{MSE}(\theta_t)$, why do we **subtract** the gradient term instead of adding it?
   * *Your Answer:* 


In the subsequent steps we will review how we can express this using `numpy`, a numeric computation module for Python that is a building block for `pandas` and `sklearn`.

In [ ]:
import numpy as np


def grad_mse(theta, X, y):
    return np.array(- 2 / len(X) * (X.T @ y - X.T @ X * theta))


thetas = [np.random.random()]
print("Random initial theta value:", thetas[-1])

We can then take a small step in the opposite direction of the gradient to roll down the hill until we converge on a good guess. To make this a machine learning algorithm, we simply put the update step in a loop until the theta values no longer noticeably change.

In [ ]:
# Run this cell multiple times to see the updates from plot_loss below

plot_line(thetas[-1])
plot_loss(thetas)

# Take a small step in the opposite direction of the gradient to roll downhill
thetas.append(thetas[-1] - 0.002 * grad_mse(thetas[-1], sensor_data[feature], sensor_data[target]))
    
thetas

### The Update Step

```python
thetas.append(thetas[-1] - 0.002 * grad_mse(thetas[-1], sensor_data["PAS"], sensor_data["AQS"]))
```

---

### Step by step explanation

#### `thetas[-1]`

This is your **current slope value** (the most recent θ in the list).

Let’s call it ( theta_t ).

---

#### `grad_mse(thetas[-1], sensor_data["PAS"], sensor_data["AQS"])`

This calls your gradient function — it computes **how the loss (MSE)** changes with respect to the slope θ at that current point.

* If the gradient is **positive**, it means your slope is *too large* (increasing θ makes the loss worse).
* If the gradient is **negative**, your slope is *too small* (you need to increase θ).

---

#### `0.002 * grad_mse(...)`

This is the **learning rate** (0.002 here).
It controls **how big a step** you take in the direction opposite the gradient.

If you take steps that are too big → you overshoot.
Too small → convergence is slow.

---

#### `thetas[-1] - (...)`

Subtracting moves **against the gradient**, i.e., downhill on the loss curve:

That’s the gradient descent rule — you’re updating θ to reduce loss.

---

#### `thetas.append(...)`

Finally, you **store the new slope value** in the list `thetas`, so you can:

* Track the history of all θ values as they evolve
* Plot them later (e.g., how θ converges)
* Use the latest one as the current slope for the next iteration

---

### Summary

That one line means:

> “Take the current slope, compute the loss gradient, step slightly downhill on the loss curve, and record the new slope.”

Each time you run it, you move a little closer to the “best-fit” slope (the bottom of the parabola from your loss surface).

## Linear regression models

What we've just described is the gradient descent algorithm for fitting a linear regression model. A **linear regression model** is a machine learning model that is used to predict a numeric value (like AQS measurements) using a linear combination of coefficients and **features** (columns from the training dataset). scikit-learn provides an easy way to do define a linear regression model, fit our training dataset `X` to the target values `y`, and examine the coefficients to look inside the model.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X = sensor_data[["PAS"]]
y = sensor_data["AQS"]
reg = LinearRegression().fit(X, y)

print("Model:", " + ".join([f"{reg.intercept_:.2f}"] + [f"{coef:.2f}({X.columns[i]})" for i, coef in enumerate(reg.coef_)]))
print("Error:", mean_squared_error(y, reg.predict(X)))
plot_line(reg.coef_[0], reg.intercept_)

This procedure is more or less how `lmplot` works!

In [ ]:
sns.lmplot(sensor_data, x="PAS", y="AQS")

But the advantage of designing our own model is that we can combine other variables to reduce the mean squared error loss. The final model that the EPA uses only takes into account the sensor measurement and the relative `humidity`, but not any other variables. Later, we'll learn why they made this decision.

In [ ]:
# Our model: predict AQS from PAS plus all the other variables
X = sensor_data[["PAS", "temp", "humidity", "dew"]]
y = sensor_data["AQS"]
reg = LinearRegression().fit(X, y)

print("Model:", " + ".join([f"{reg.intercept_:.2f}"] + [f"{coef:.2f}({X.columns[i]})" for i, coef in enumerate(reg.coef_)]))
print("Error:", mean_squared_error(y, reg.predict(X)))

In [ ]:
# The EPA's simpler model: predict AQS from PAS and humidity only
X = sensor_data[["PAS", "humidity"]]
y = sensor_data["AQS"]
reg = LinearRegression().fit(X, y)

print("Model:", " + ".join([f"{reg.intercept_:.2f}"] + [f"{coef:.2f}({X.columns[i]})" for i, coef in enumerate(reg.coef_)]))
print("Error:", mean_squared_error(y, reg.predict(X)))

## Reflection

**Parameters vs. Hyperparameters:** 
   * Identify which of the following is a **Model Parameter** ($\theta$) and which is a **Hyperparameter** ($\alpha$):
     * The slope value adjusted automatically during gradient descent loop iterations: ___________
     * The step size multiplier chosen manually by the programmer before training: ___________
   * Briefly explain why this distinction matters when building machine learning models.
   * *Your Answer:* 

**Gradient Sign:** If the calculated gradient at your current slope guess is **positive**, should the algorithm **increase** or **decrease** the slope value ($\theta$) on the next iteration? Explain why.
   * *Your Answer:* 

**Manual vs. Automated Optimization:** Compare your best manual slope guess from the "Guessing Game" to the final slope you reach after running the gradient descent update cell many times. Did gradient descent achieve a lower Mean Squared Error (MSE)? Why is automated optimization essential when scaling to datasets with 10+ features?
   * *Your Answer:* 